## CKD and Diabetes prediction with Explainable AI

### Notebook for the Machine Learning Model for the prediction with XAI

This notebook has the trained Machine Learning Model with the results for Chronic Kidney Disease and Diabetes. Results are not just predictions it has the explanation for the prediction. Prediction are not coming from a blackbox

In [1]:
#Importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import shap
import lime
import lime.lime_tabular

d:\projects\CKD_and_Diabetes_prediction_with_Explainable_AI_-XAI-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Data Import and Pre process

In [10]:
#Importing Dataset

df = pd.read_csv('dataset/kidney_disease.csv')

df.head()

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35,7300,4.6,no,no,no,good,no,no,ckd


In [ ]:
#Shape of the table
print("Shape: ", df.shape)

# 26 Features and 400 instances

#Summary statistics
print("Summary Statistics: ", df.describe())

Shape:  (400, 26)
Summary Statistics:                 id         age          bp  ...         sod         pot        hemo
count  400.000000  391.000000  388.000000  ...  313.000000  312.000000  348.000000
mean   199.500000   51.483376   76.469072  ...  137.528754    4.627244   12.526437
std    115.614301   17.169714   13.683637  ...   10.408752    3.193904    2.912587
min      0.000000    2.000000   50.000000  ...    4.500000    2.500000    3.100000
25%     99.750000   42.000000   70.000000  ...  135.000000    3.800000   10.300000
50%    199.500000   55.000000   80.000000  ...  138.000000    4.400000   12.650000
75%    299.250000   64.500000   80.000000  ...  142.000000    4.900000   15.000000
max    399.000000   90.000000  180.000000  ...  163.000000   47.000000   17.800000

[8 rows x 12 columns]


In [22]:
#Data preprocess

#Replacing '?' with NaN
df.replace('?', np.nan, inplace=True)

#convert columns to numeric 
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

#drop rows with too many missing values
df = df.dropna()

# #Filling missing values with mean for numerical columns
# for col in df.select_dtypes(include=[np.number]).columns:
#     df[col].fillna(df[col].mean(), inplace=True)

#Encode the categorical variables
for col in df.select_dtypes(include=['object']).columns:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop("classification", axis=1)
y = df["classification"]

#Splitting the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

#scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

C:\Users\Anushka\AppData\Local\Temp\ipykernel_19248\514933059.py:8: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [23]:
df

,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
3,3,48.0,70.0,1.005,4.0,0.0,1,0,1,0,117.0,56.0,3.8,111.0,2.5,11.2,32,6700,3.9,1,0,0,1,1,1,0
9,9,53.0,90.0,1.020,2.0,0.0,0,0,1,0,70.0,107.0,7.2,114.0,3.7,9.5,29,12100,3.7,1,1,0,1,0,1,0
11,11,63.0,70.0,1.010,3.0,0.0,0,0,1,0,380.0,60.0,2.7,131.0,4.2,10.8,32,4500,3.8,1,1,0,1,1,0,0
14,14,68.0,80.0,1.010,3.0,2.0,1,0,1,1,157.0,90.0,4.1,130.0,6.4,5.6,16,11000,2.6,1,1,1,1,1,0,0
20,20,61.0,80.0,1.015,2.0,0.0,0,0,0,0,173.0,148.0,3.9,135.0,5.2,7.7,24,9200,3.2,1,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,395,55.0,80.0,1.020,0.0,0.0,1,1,0,0,140.0,49.0,0.5,150.0,4.9,15.7,47,6700,4.9,0,0,0,0,0,0,1
396,396,42.0,70.0,1.025,0.0,0.0,1,1,0,0,75.0,31.0,1.2,141.0,3.5,16.5,54,7800,6.2,0,0,0,0,0,0,1
397,397,12.0,80.0,1.020,0.0,0.0,1,1,0,0,100.0,26.0,0.6,137.0,4.4,15.8,49,6600,5.4,0,0,0,0,0,0,1
398,398,17.0,60.0,1.025,0.0,0.0,1,1,0,0,114.0,50.0,1.0,135.0,4.9,14.2,51,7200,5.9,0,0,0,0,0,0,1
